# 2. Tải, lọc và kiểm toán nguồn

## 2.1 Tải riêng từng version

Downloader gọi endpoint `/src/<paper>vN` trên mirror export. Tên đích gắn với revision đã
yêu cầu; không tải latest rồi gán nội dung đó vào một version cũ. Mặc định source và metadata
chờ 3 giây giữa các request arXiv; số đo lịch sử vẫn ghi đúng cấu hình của lượt lịch sử đó.

Kết nối được tái sử dụng nhưng các lượt truyền source được tuần tự hóa. Archive tải mới
được đọc theo luồng. Khi có `.part`, Range request tiếp tục từ byte đã có và phải khớp
`Content-Range`; khi phản hồi không thể ghép an toàn, chương trình không nối mù hai phần.
Resume dùng đoạn tăng dần 1–4–8 MiB khi ổn định và giảm lại khi lỗi.

## 2.2 Nhận diện và giữ nội dung

Phản hồi không phải lúc nào cũng là tar.gz. Chương trình kiểm tra dạng tar, gzip, TeX hoặc
PDF bằng nội dung. Corpus giữ `.tex`, `.bib` và `.bbl` bổ trợ theo QA, không chỉnh câu lệnh
figure trong TeX. Vì vậy M2 vẫn nhận diện được vị trí figure/table dù binary ảnh đã bị loại.

Việc giải nén đi qua staging, kiểm tra đường dẫn và symlink, rồi mới thay nguồn cũ sau khi
kết quả mới hợp lệ. Nếu tải hoặc giải nén lỗi, trạng thái lỗi và phần có thể resume được giữ.

Giữ `.bbl` là điều chỉnh theo QA sau các lượt đo đang được trình bày. Parser dùng nó khi
bibliography gốc thiếu hoặc không đọc được và đường dẫn biên dịch xác định được file đó.
Snapshot lịch sử chưa được tải lại để phục hồi `.bbl` đã bị lọc; các số đo giữ nguyên.

| Trạng thái version | Bằng chứng và cách dùng |
|---|---|
| success | Có TeX, file đúng allowlist và dung lượng khớp record khi audit |
| pdf_only / no_latex | Có manifest xác nhận loại nguồn; M2 ghi empty |
| error / pending / chưa biết | Cần retry hoặc xác minh; không chuyển thành empty |

## 2.3 Audit độc lập

Audit không chỉ đọc counter `success`: nó tìm đúng thư mục version, đếm TeX/Bib, kiểm tra
file ngoài allowlist, đối chiếu size và manifest của các trường hợp empty. Những version
không đạt được đưa vào retry manifest. Dưới đây là kết quả trên snapshot hiện tại.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
audit = pd.Series(m1['source_audit'], name='versions')
display(audit.to_frame())
audit.plot.bar(title='Kết quả audit nguồn', ylabel='Số version'); plt.tight_layout(); plt.show()
rates = pd.DataFrame([
    {'metric':'TeX đã xác minh / mọi version', 'numerator':m1['source_audit'].get('tex_verified',0), 'denominator':scope['versions']},
    {'metric':'Nguồn hợp lệ gồm empty / mọi version', 'numerator':m1['source_verified'], 'denominator':scope['versions']},
    {'metric':'Paper có mọi nguồn đã xác minh', 'numerator':m1.get('papers_with_all_sources_verified'), 'denominator':scope['papers']},
    {'metric':'Paper đủ nguồn và có S2 success/empty', 'numerator':m1.get('papers_with_sources_and_s2_verified'), 'denominator':scope['papers']},
])
rates['ratio'] = rates['numerator'] / rates['denominator']; display(rates)

## 2.4 Kích thước trước và sau lọc

Gọi $E$ là tập paper có mọi version TeX đã được audit. Với mỗi paper, cộng byte của các
version trước khi lấy trung bình. Cả hai trung bình dùng cùng tập $E$:


$$

\overline{B}_{\mathrm{before}}=\frac{\sum_{p\in E}\sum_v B_{p,v,\mathrm{before}}}{|E|},\qquad
\overline{B}_{\mathrm{after}}=\frac{\sum_{p\in E}\sum_v B_{p,v,\mathrm{after}}}{|E|}.

$$


Tỷ lệ giảm byte được tính từ hai tổng trên cùng tập. `before` là byte nội dung giải nén
trước lọc, không phải byte archive nén hoặc dung lượng mạng thực truyền. `after` là byte
TeX/Bib được giữ. Tỷ lệ giảm này mô tả lọc nội dung, không mô tả tỷ lệ nén gzip.

In [ ]:
sizes = pd.DataFrame([{'eligible_papers':m1['eligible_size_papers'],
                      'mean_before_bytes':m1['mean_before_bytes'], 'mean_after_bytes':m1['mean_after_bytes'],
                      'reduction_fraction':m1['source_reduction']}]); display(sizes)
pd.Series({'Before filtering':m1['mean_before_bytes'], 'TeX/Bib retained':m1['mean_after_bytes']}).plot.bar(
    ylabel='Byte/paper đủ điều kiện', title='Kích thước trung bình trên cùng tập paper')
plt.tight_layout(); plt.show()
display(Markdown(f"Trên **{m1['eligible_size_papers']} paper** đủ điều kiện, byte giữ lại giảm "
                 f"**{m1['source_reduction']:.2%}**. Version empty và paper còn lỗi không nằm trong mẫu số này."))

## 2.5 Resume, Drive backup và phần còn thiếu

Trên Colab, source và shard làm việc ở `/content`; archive backup đã xác minh được sao lưu
theo lô lên Drive. Checkpoint ghi nhận các part đã commit. Khi restore, cần kiểm tra hash và
chuỗi generation trước khi xem một workspace là bản khôi phục toàn bộ. Snapshot của báo
cáo chỉ dùng một số lô, nên được ghi rõ là khôi phục một phần.

Mỗi checkpoint/backup chỉ có một runtime ghi. `LocalMutex` chỉ điều phối các tiến trình
trên cùng máy, không tạo khóa phân tán giữa nhiều Colab. Bổ sung số runtime không thay cho
việc tuân thủ giới hạn request chung hoặc bảo vệ checkpoint.

Version nguồn còn lỗi giữ nguyên trong danh sách cần xử lý. Không tải thêm toàn dải trong
lần chuẩn bị báo cáo này; dữ liệu nguồn của snapshot được giữ để M2 xử lý và đối chiếu.

## Tài liệu và mã tham chiếu

- [Đặc tả M1](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/crawler/source_downloader.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/source_downloader.py).
- Mã triển khai: [src/crawler/validate_sources.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/validate_sources.py).
- Mã triển khai: [src/crawler/retry_sources.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/retry_sources.py).
- Mã triển khai: [src/crawler/drive_backup.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/crawler/drive_backup.py).